In [1]:
# --- Workspace Verification ---
# This cell loads the artifacts from your previous notebook to ensure 
# the training pipeline has all necessary components in memory.

import torch
import pandas as pd
import pickle
import joblib
from pathlib import Path
from torch_geometric.data import Data

# 1. Setup Paths
root = Path("/home/jupyter-1nt23cb058/Capstone")
tensor_dir = root / "data/processed/tensors"
graph_path = root / "data/processed/graph/topology_graph.pt"
scaler_path = root / "data/processed/graph/global_scaler.pkl"

print("Checking artifacts...")

# 2. Load the Graph
# Note: We handled the 'dict' vs 'Data' object issue in the previous notebook.
graph_data = torch.load(graph_path)
if isinstance(graph_data, dict) and 'edge_index' in graph_data:
    # Reconstruct from dictionary if necessary
    edge_index = graph_data['edge_index']
    num_nodes = graph_data.get('num_nodes', edge_index.max().item() + 1)
    graph = Data(edge_index=edge_index, num_nodes=num_nodes)
else:
    graph = graph_data

# 3. Load Tensors
X_train = torch.load(tensor_dir / "X_train.pt")
y_train = torch.load(tensor_dir / "y_train.pt")
node_ids = torch.load(tensor_dir / "node_ids.pt")

# 4. Load Scaler
with open(scaler_path, 'rb') as f:
    scaler = joblib.load(f)

# 5. Device Setup (V100 check)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- Output Report ---
print("-" * 30)
print(f"Device:          {device}")
print(f"X_train Shape:   {X_train.shape} (Sequences, Time, Features)")
print(f"y_train Shape:   {y_train.shape}")
print(f"Graph Nodes:     {graph.num_nodes:,}")
print(f"Graph Edges:     {graph.edge_index.shape[1]:,}")
print(f"Target Mean:     {scaler['target']['mean']:.4f}")
print("-" * 30)
print("READY FOR MODEL ARCHITECTURE.")

Checking artifacts...
------------------------------
Device:          cuda
X_train Shape:   torch.Size([1069795, 24, 17]) (Sequences, Time, Features)
y_train Shape:   torch.Size([1069795, 1])
Graph Nodes:     154,902
Graph Edges:     424,550
Target Mean:     39.2206
------------------------------
READY FOR MODEL ARCHITECTURE.


In [9]:
# HOTFIX - remove Python 3.10-only zip(strict=...) usage
# This allows the GNN model to run on environments with Python < 3.10
from pathlib import Path

model_file = Path('/home/jupyter-1nt23cb058/Capstone/gnn/model.py')
txt = model_file.read_text(encoding='utf-8')

old = 'for conv, norm in zip(self.gnn_layers, self.norms, strict=False):'
new = 'for conv, norm in zip(self.gnn_layers, self.norms):'

if old not in txt:
    print('Target line not found. Checking for any strict= usage...')
    for i, line in enumerate(txt.splitlines(), 1):
        if 'strict=' in line:
            print(f'Line {i}: {line}')
else:
    txt = txt.replace(old, new)
    model_file.write_text(txt, encoding='utf-8')
    print('Patched:', model_file)

# Reload module so notebook uses patched code without restarting kernel
import importlib
import gnn.model as gnn_model
importlib.reload(gnn_model)

from gnn.model import STPIGNN, train_step_amp, masked_data_loss
print('Reloaded gnn.model successfully')

Patched: /home/jupyter-1nt23cb058/Capstone/gnn/model.py
Reloaded gnn.model successfully


In [2]:
# PHASE 1 - Environment and import setup
import os
import sys
import copy
import math
import random
import numpy as np
from pathlib import Path

project_root = root
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from gnn.model import STPIGNN, train_step_amp, masked_data_loss

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False

print('Imports OK, cwd=', Path.cwd())
print('PyTorch version:', torch.__version__)

Imports OK, cwd= /home/jupyter-1nt23cb058/Capstone
PyTorch version: 2.8.0+cu128


In [3]:
# PHASE 2 - Imports, deterministic setup, and graph load
import os
import sys
import copy
import random
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

project_root = root
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from gnn.model import STPIGNN, train_step_amp, masked_data_loss

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False

pyg_graph_path = root / 'data/processed/graph/topology_graph_pyg_inference.pt'
if pyg_graph_path.exists():
    graph_pyg = torch.load(pyg_graph_path, weights_only=False)
    edge_index = graph_pyg.edge_index.long()
    edge_attr = graph_pyg.edge_attr.float() if hasattr(graph_pyg, 'edge_attr') and graph_pyg.edge_attr is not None else torch.ones(edge_index.shape[1], 1)
    num_nodes = int(graph_pyg.num_nodes)
    if hasattr(graph_pyg, 'train_mask') and graph_pyg.train_mask is not None:
        train_mask = graph_pyg.train_mask.bool().cpu()
    else:
        train_mask = torch.zeros(num_nodes, dtype=torch.bool)
else:
    edge_index = graph.edge_index.long().cpu()
    edge_attr = torch.ones(edge_index.shape[1], 1, dtype=torch.float32)
    num_nodes = int(graph.num_nodes)
    train_mask = torch.load(tensor_dir / 'train_mask.pt').bool() if (tensor_dir / 'train_mask.pt').exists() else torch.zeros(num_nodes, dtype=torch.bool)

upwind_path = tensor_dir / 'upwind_edge_mask.pt'
upwind_edge_mask = torch.load(upwind_path).bool() if upwind_path.exists() else None

print({'torch': torch.__version__, 'device': str(device), 'num_nodes': num_nodes, 'num_edges': int(edge_index.shape[1])})
print({'train_mask_active': int(train_mask.sum().item()), 'train_mask_total': int(train_mask.numel()), 'upwind_edge_mask_present': upwind_edge_mask is not None})

{'torch': '2.8.0+cu128', 'device': 'cuda', 'num_nodes': 154902, 'num_edges': 424550}
{'train_mask_active': 23, 'train_mask_total': 154902, 'upwind_edge_mask_present': False}


In [5]:
# PHASE 3: Memory-Safe Subgraph Construction
# This notebook builds a local subgraph around sensor nodes to prevent RAM exhaustion when handling 150k+ nodes.

# PHASE 3A - Build memory-safe local subgraph around sensor nodes
import torch
import numpy as np
import pandas as pd
from torch_geometric.utils import k_hop_subgraph

if 'train_mask' not in globals():
    train_mask_path = tensor_dir / 'train_mask.pt'
    train_mask = torch.load(train_mask_path).bool()

sensor_idx = torch.where(train_mask)[0].long()
if sensor_idx.numel() == 0:
    raise RuntimeError('No sensor nodes found in train_mask')

# Tune these if needed
NUM_HOPS = 2
MAX_LOCAL_NODES = 8000

subset, edge_index_local, mapping, edge_mask = k_hop_subgraph(
    node_idx=sensor_idx,
    num_hops=NUM_HOPS,
    edge_index=edge_index.cpu(),
    relabel_nodes=True,
    num_nodes=int(num_nodes),
)

if subset.numel() > MAX_LOCAL_NODES:
    keep = torch.randperm(subset.numel())[:MAX_LOCAL_NODES]
    keep_nodes = subset[keep].sort().values
    subset, edge_index_local, _, edge_mask = k_hop_subgraph(
        node_idx=keep_nodes,
        num_hops=0,
        edge_index=edge_index.cpu(),
        relabel_nodes=True,
        num_nodes=int(num_nodes),
    )

edge_attr_local = edge_attr.cpu()[edge_mask] if edge_attr is not None else torch.ones(edge_index_local.shape[1], 1)

global_to_local = {int(g): i for i, g in enumerate(subset.tolist())}
sensor_global = set(sensor_idx.tolist())
local_sensor_ids = [global_to_local[g] for g in sensor_global if g in global_to_local]

train_mask_local = torch.zeros(subset.numel(), dtype=torch.bool)
if len(local_sensor_ids) > 0:
    train_mask_local[torch.tensor(local_sensor_ids, dtype=torch.long)] = True

print({
    'global_nodes': int(num_nodes),
    'local_nodes': int(subset.numel()),
    'local_edges': int(edge_index_local.shape[1]),
    'local_sensor_count': int(train_mask_local.sum().item())
})

# PHASE 3B - Build window tensors on local nodes only (fixed shape-safe version)
import pandas as pd
import numpy as np
import torch

model_input_path = root / 'data/processed/model_input/model_input_node_hourly_features.parquet'
node_map_path = root / 'data/processed/graph/topology_nodeid_to_index_map.parquet'

df = pd.read_parquet(model_input_path)
node_map = pd.read_parquet(node_map_path)
node_to_idx = dict(zip(node_map['node_id'].values, node_map['node_index'].values))

feature_cols = [
    'station_pm10', 'station_pm25', 'station_no2', 'station_so2', 'station_co',
    'weather_wind_speed_10m', 'weather_wind_direction_10m', 'weather_wind_gusts_10m',
    'weather_temperature_2m', 'weather_relative_humidity_2m', 'weather_surface_pressure',
    'city_nitrogen_dioxide', 'city_sulphur_dioxide', 'city_pm2_5', 'city_pm10', 'city_carbon_monoxide'
]
target_col = 'station_pm25'
feature_cols = [c for c in feature_cols if c in df.columns]

if len(feature_cols) == 0:
    raise ValueError('No feature columns available in model_input parquet')
if target_col not in df.columns:
    raise ValueError('Target column station_pm25 not found')

df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
df = df.dropna(subset=['timestamp', 'node_id']).copy()
df['node_id'] = pd.to_numeric(df['node_id'], errors='coerce').astype('Int64')
df = df.dropna(subset=['node_id']).copy()
df['node_id'] = df['node_id'].astype('int64')
df['node_index'] = df['node_id'].map(node_to_idx)
df = df.dropna(subset=['node_index']).copy()
df['node_index'] = df['node_index'].astype('int64')

# Keep only local subgraph nodes (subset defined in Phase 3A)
local_global_nodes = set(int(x) for x in subset.tolist())
df = df[df['node_index'].isin(local_global_nodes)].copy()

# Optional downsampling for trial speed/memory
LOOKBACK_DAYS = 90
TIME_STRIDE_HOURS = 1
tmax = df['timestamp'].max()
df = df[df['timestamp'] >= (tmax - pd.Timedelta(days=LOOKBACK_DAYS))].copy()

if TIME_STRIDE_HOURS > 1:
    base = df['timestamp'].min()
    hours = ((df['timestamp'] - base).dt.total_seconds() // 3600).astype('int64')
    df = df[(hours % TIME_STRIDE_HOURS) == 0].copy()

# Map global node index to local node index
global_to_local = {int(g): i for i, g in enumerate(subset.tolist())}
df['node_local'] = df['node_index'].map(global_to_local)
df = df.dropna(subset=['node_local']).copy()
df['node_local'] = df['node_local'].astype('int64')

all_times = pd.Index(sorted(df['timestamp'].dropna().unique()))
N_local = int(subset.numel())
full_local_nodes = np.arange(N_local, dtype=np.int64)

if len(all_times) < 20:
    raise ValueError(f'Not enough time steps after filtering: {len(all_times)}')

# Build X safely as stack of [T, N] matrices per feature
feature_mats = []
for c in feature_cols:
    p = df.pivot_table(index='timestamp', columns='node_local', values=c, aggfunc='mean')
    p = p.reindex(index=all_times, columns=full_local_nodes)
    p = p.astype('float32').ffill().bfill().fillna(0.0)
    feature_mats.append(p.to_numpy(dtype='float32'))

x_vals = np.stack(feature_mats, axis=-1)  # [Time, Nodes, Features]

# Build y as [Time, Nodes]
p_y = df.pivot_table(index='timestamp', columns='node_local', values=target_col, aggfunc='mean')
p_y = p_y.reindex(index=all_times, columns=full_local_nodes)
p_y = p_y.astype('float32').ffill().bfill().fillna(0.0)
y_vals = p_y.to_numpy(dtype='float32')

x_time_node_feat = torch.tensor(x_vals, dtype=torch.float32)
y_time_node = torch.tensor(y_vals, dtype=torch.float32)

# Export training graph vars expected by next phases
edge_index_train = edge_index_local.long()
edge_attr_train = edge_attr_local.float()
train_mask_train = train_mask_local.bool()
num_nodes_train = N_local

print({
    'time_steps': int(x_time_node_feat.shape[0]),
    'num_nodes_train': int(num_nodes_train),
    'feature_dim': int(x_time_node_feat.shape[2]),
    'x_shape': tuple(x_time_node_feat.shape),
    'y_shape': tuple(y_time_node.shape),
    'sensor_nodes_train': int(train_mask_train.sum().item())
})

# PHASE 3C - Compatibility aliases for later phases
edge_index = edge_index_train
edge_attr = edge_attr_train
train_mask = train_mask_train
num_nodes = int(num_nodes_train)
print({'num_nodes_for_training': num_nodes, 'num_edges_for_training': int(edge_index.shape[1])})

{'global_nodes': 154902, 'local_nodes': 183, 'local_edges': 390, 'local_sensor_count': 23}
{'time_steps': 2161, 'num_nodes_train': 183, 'feature_dim': 16, 'x_shape': (2161, 183, 16), 'y_shape': (2161, 183), 'sensor_nodes_train': 23}
{'num_nodes_for_training': 183, 'num_edges_for_training': 390}


In [6]:

# PHASE 4 - Build window dataset and temporal split
from torch.utils.data import Dataset, DataLoader

WINDOW = 12

class WindowDataset(Dataset):
    def __init__(self, x_tnf, y_tn, start_t, end_t, window):
        self.x = x_tnf  # [T, N, F]
        self.y = y_tn   # [T, N]
        self.window = window
        self.indices = list(range(start_t, end_t - window))
        if len(self.indices) <= 0:
            raise ValueError('No windows available in requested split')

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        t0 = self.indices[i]
        x_seq = self.x[t0:t0 + self.window]      # [W, N, F]
        y_next = self.y[t0 + self.window]        # [N]
        return x_seq, y_next

T_total = x_time_node_feat.shape[0]
split_t = int(0.80 * T_total)

train_ds = WindowDataset(x_time_node_feat, y_time_node, start_t=0, end_t=split_t, window=WINDOW)
val_ds = WindowDataset(x_time_node_feat, y_time_node, start_t=split_t - WINDOW, end_t=T_total, window=WINDOW)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

print({'train_windows': len(train_ds), 'val_windows': len(val_ds), 'window': WINDOW})

{'train_windows': 1716, 'val_windows': 433, 'window': 12}


In [11]:
# PHASE 5 - Initialize ST-PIGNN on local subgraph
import copy
import numpy as np
import torch
from gnn.model import STPIGNN, train_step_amp, masked_data_loss

model = STPIGNN(
    node_in_dim=int(x_time_node_feat.shape[-1]),
    edge_dim=int(edge_attr.shape[-1]),
    spatial_hidden_dim=96,
    temporal_hidden_dim=96,
    gnn_layers=2,
    gru_layers=1,
    dropout=0.15,
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=5e-4)
scaler_amp = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

edge_index_dev = edge_index.long().to(device)
edge_attr_dev = edge_attr.float().to(device)
train_mask_dev = train_mask.bool().to(device)

# upwind is optional; if missing we run clean data-only baseline
upwind_mask_dev = upwind_edge_mask.bool().to(device) if ('upwind_edge_mask' in globals() and upwind_edge_mask is not None) else None
USE_PHYSICS = bool(upwind_mask_dev is not None)

def lambda_phys_schedule(epoch, warmup_epochs=8, max_lambda=0.25):
    if not USE_PHYSICS:
        return 0.0
    if epoch < warmup_epochs:
        return max_lambda * float(epoch + 1) / float(warmup_epochs)
    return max_lambda

print({'physics_enabled': USE_PHYSICS, 'mask_active': int(train_mask.sum().item()), 'nodes_train': int(num_nodes)})

{'physics_enabled': False, 'mask_active': 23, 'nodes_train': 183}


In [13]:
# PHASE 6A - Inspect target scale on supervised nodes only
m = train_mask.bool()
y_supervised = y_time_node[:, m]  # [T, sensors]

q = torch.quantile(y_supervised.flatten(), torch.tensor([0.50, 0.90, 0.95, 0.99], dtype=torch.float32))
print({
    "y_min": float(y_supervised.min()),
    "y_max": float(y_supervised.max()),
    "q50": float(q[0]),
    "q90": float(q[1]),
    "q95": float(q[2]),
    "q99": float(q[3]),
})

{'y_min': 0.0, 'y_max': 373.6192626953125, 'q50': 39.032501220703125, 'q90': 147.07936096191406, 'q95': 224.6432342529297, 'q99': 373.6192626953125}


In [14]:
# PHASE 6B - Rescale target for training stability
# Use q95 as robust scale; fallback to 100 if degenerate
TARGET_SCALE = float(max(50.0, torch.quantile(y_supervised.flatten(), 0.95).item()))
print({"TARGET_SCALE": TARGET_SCALE})

y_time_node_scaled = y_time_node / TARGET_SCALE

{'TARGET_SCALE': 224.6432342529297}


In [15]:
# PHASE 6C - Rebuild datasets with scaled target (same windows/split)
train_ds = WindowDataset(x_time_node_feat, y_time_node_scaled, start_t=0, end_t=split_t, window=WINDOW)
val_ds = WindowDataset(x_time_node_feat, y_time_node_scaled, start_t=split_t - WINDOW, end_t=T_total, window=WINDOW)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)
print({"train_windows": len(train_ds), "val_windows": len(val_ds), "target_scaled": True})

{'train_windows': 1716, 'val_windows': 433, 'target_scaled': True}


In [21]:
# PHASE 5A - Build upwind_edge_mask for current local training graph
import numpy as np
import pandas as pd
import torch

required = ['root', 'subset', 'edge_index', 'num_nodes']
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError(f'Missing prerequisites for upwind build: {missing}')

node_map_path = root / 'data/processed/graph/topology_nodeid_to_index_map.parquet'
nodes_path = root / 'data/graphs/bangalore_utm_nodes.parquet'
model_input_path = root / 'data/processed/model_input/model_input_node_hourly_features.parquet'

node_map = pd.read_parquet(node_map_path)
nodes_df = pd.read_parquet(nodes_path)
model_input = pd.read_parquet(model_input_path)

if 'osmid' not in nodes_df.columns:
    raise ValueError('Expected osmid column in nodes parquet')

x_col = 'x' if 'x' in nodes_df.columns else ('utm_x' if 'utm_x' in nodes_df.columns else None)
y_col = 'y' if 'y' in nodes_df.columns else ('utm_y' if 'utm_y' in nodes_df.columns else None)
if x_col is None or y_col is None:
    raise ValueError('Could not find coordinate columns (x/utm_x and y/utm_y) in nodes parquet')

idx_to_nodeid = dict(zip(node_map['node_index'].values, node_map['node_id'].values))
nodeid_to_xy = dict(zip(nodes_df['osmid'].astype('int64').values, nodes_df[[x_col, y_col]].astype('float64').itertuples(index=False, name=None)))

def circular_mean_deg(series_deg):
    r = np.deg2rad(series_deg.to_numpy(dtype='float64'))
    s = np.sin(r).mean()
    c = np.cos(r).mean()
    ang = (np.rad2deg(np.arctan2(s, c)) + 360.0) % 360.0
    return float(ang)

# Extract Wind Direction
if 'weather_wind_direction_10m' in model_input.columns:
    wd = pd.to_numeric(model_input['weather_wind_direction_10m'], errors='coerce').dropna()
    if len(wd) == 0:
        raise ValueError('weather_wind_direction_10m exists but has no numeric values')
    wind_dir = circular_mean_deg(wd)
else:
    u_candidates = [c for c in model_input.columns if c.lower() in ['weather_wind_u_10m', 'wind_u', 'u10', 'u_component']]
    v_candidates = [c for c in model_input.columns if c.lower() in ['weather_wind_v_10m', 'wind_v', 'v10', 'v_component']]
    if len(u_candidates) == 0 or len(v_candidates) == 0:
        raise ValueError('No usable wind direction or u/v columns found in model_input')
    u = pd.to_numeric(model_input[u_candidates[0]], errors='coerce').dropna()
    v = pd.to_numeric(model_input[v_candidates[0]], errors='coerce').dropna()
    wind_dir = float((np.degrees(np.arctan2(v.mean(), u.mean())) + 360.0) % 360.0)

UPWIND_THRESHOLD_DEG = 110.0

src_local = edge_index[0].detach().cpu().numpy()
dst_local = edge_index[1].detach().cpu().numpy()

mask = np.zeros(len(src_local), dtype=bool)
missing_xy = 0
usable = 0

# Geometry logic: determine if destination is upwind of source
for i in range(len(src_local)):
    s_loc = int(src_local[i])
    d_loc = int(dst_local[i])

    s_global_idx = int(subset[s_loc].item()) if 'subset' in globals() else s_loc
    d_global_idx = int(subset[d_loc].item()) if 'subset' in globals() else d_loc

    s_nodeid = idx_to_nodeid.get(s_global_idx)
    d_nodeid = idx_to_nodeid.get(d_global_idx)
    if s_nodeid is None or d_nodeid is None:
        missing_xy += 1
        continue

    s_xy = nodeid_to_xy.get(int(s_nodeid))
    d_xy = nodeid_to_xy.get(int(d_nodeid))
    if s_xy is None or d_xy is None:
        missing_xy += 1
        continue

    dx = float(d_xy[0] - s_xy[0])
    dy = float(d_xy[1] - s_xy[1])
    if abs(dx) < 1e-8 and abs(dy) < 1e-8:
        continue

    edge_bearing = (np.degrees(np.arctan2(dy, dx)) + 360.0) % 360.0
    delta = abs(((edge_bearing - wind_dir + 180.0) % 360.0) - 180.0)

    # If edge direction opposes wind direction enough, destination is treated as upwind
    mask[i] = bool(delta >= UPWIND_THRESHOLD_DEG)
    usable += 1

upwind_edge_mask = torch.from_numpy(mask)
torch.save(upwind_edge_mask, tensor_dir / 'upwind_edge_mask.pt')

print({
    'wind_dir_deg': round(wind_dir, 2),
    'threshold_deg': UPWIND_THRESHOLD_DEG,
    'edges_total': int(len(mask)),
    'edges_usable': int(usable),
    'edges_missing_xy': int(missing_xy),
    'upwind_true': int(upwind_edge_mask.sum().item())
})


# PHASE 5B - Confirm physics mode before training
upwind_edge_mask = torch.load(tensor_dir / 'upwind_edge_mask.pt').bool()
upwind_mask_dev = upwind_edge_mask.to(device)

USE_PHYSICS = True

def lambda_phys_schedule(epoch, warmup_epochs=6, max_lambda=0.15):
    if epoch < warmup_epochs:
        return max_lambda * float(epoch + 1) / float(warmup_epochs)
    return max_lambda

print({
    'physics_enabled': USE_PHYSICS,
    'upwind_edge_count': int(upwind_edge_mask.sum().item()),
    'lambda_epoch0': round(lambda_phys_schedule(0), 4),
    'lambda_epoch6': round(lambda_phys_schedule(6), 4)
})

{'wind_dir_deg': 84.99, 'threshold_deg': 110.0, 'edges_total': 390, 'edges_usable': 390, 'edges_missing_xy': 0, 'upwind_true': 136}
{'physics_enabled': True, 'upwind_edge_count': 136, 'lambda_epoch0': 0.025, 'lambda_epoch6': 0.15}


In [22]:
#phase 5
import copy
import numpy as np
import torch
from gnn.model import STPIGNN, train_step_amp, masked_data_loss
from torch.utils.data import DataLoader

required = [
    'x_time_node_feat', 'y_time_node', 'train_mask', 'edge_index', 'edge_attr',
    'train_ds', 'val_ds', 'WINDOW', 'split_t', 'T_total', 'device'
]
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError(f'Missing prerequisites before Phase 5: {missing}')

# Robust target scaling on supervised nodes
m = train_mask.bool()
y_supervised = y_time_node[:, m]
# We use the 95th percentile to scale, ensuring most values fall in [0, 1] 
# while capping the scale at a minimum of 50.0 to prevent division by zero/small numbers.
TARGET_SCALE = float(max(50.0, torch.quantile(y_supervised.flatten(), 0.95).item()))

y_time_node_scaled = y_time_node / TARGET_SCALE

# Rebuild datasets/loaders with scaled target
train_ds = WindowDataset(x_time_node_feat, y_time_node_scaled, start_t=0, end_t=split_t, window=WINDOW)
val_ds = WindowDataset(x_time_node_feat, y_time_node_scaled, start_t=split_t - WINDOW, end_t=T_total, window=WINDOW)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

model = STPIGNN(
    node_in_dim=int(x_time_node_feat.shape[-1]),
    edge_dim=int(edge_attr.shape[-1]),
    spatial_hidden_dim=96,
    temporal_hidden_dim=96,
    gnn_layers=2,
    gru_layers=1,
    dropout=0.15,
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=5e-4)
scaler_amp = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

edge_index_dev = edge_index.long().to(device)
edge_attr_dev = edge_attr.float().to(device)
train_mask_dev = train_mask.bool().to(device)

upwind_mask_dev = upwind_edge_mask.bool().to(device) if ('upwind_edge_mask' in globals() and upwind_edge_mask is not None) else None
USE_PHYSICS = bool(upwind_mask_dev is not None)

def lambda_phys_schedule(epoch, warmup_epochs=8, max_lambda=0.25):
    if not USE_PHYSICS:
        return 0.0
    if epoch < warmup_epochs:
        return max_lambda * float(epoch + 1) / float(warmup_epochs)
    return max_lambda

print({
    'physics_enabled': USE_PHYSICS,
    'target_scale_q95': TARGET_SCALE,
    'train_windows': len(train_ds),
    'val_windows': len(val_ds),
    'mask_active': int(train_mask.sum().item()),
    'nodes_train': int(train_mask.numel())
})



{'physics_enabled': True, 'target_scale_q95': 224.6432342529297, 'train_windows': 1716, 'val_windows': 433, 'mask_active': 23, 'nodes_train': 183}


In [23]:
# PHASE 6 - Train with early stopping (scaled loss, unscaled metrics)
@torch.no_grad()
def eval_masked_loss_scaled(model, loader):
    model.eval()
    vals = []
    for xb, yb_scaled in loader:
        xb = xb.to(device, non_blocking=True)
        yb_scaled = yb_scaled.to(device, non_blocking=True)
        pred_scaled = model(x_seq=xb, edge_index=edge_index_dev, edge_attr=edge_attr_dev)
        loss = masked_data_loss(pred=pred_scaled, target=yb_scaled, train_mask=train_mask_dev)
        vals.append(float(loss.detach().cpu().item()))
    return float(np.mean(vals)) if vals else float('nan')

@torch.no_grad()
def eval_masked_mae_rmse_unscaled(model, loader, scale):
    model.eval()
    m = train_mask_dev.view(1, -1)
    ae, se = [], []
    for xb, yb_scaled in loader:
        xb = xb.to(device, non_blocking=True)
        # Unscale the target for real-world metric evaluation
        yb = yb_scaled.to(device, non_blocking=True) * scale
        pred_scaled = model(x_seq=xb, edge_index=edge_index_dev, edge_attr=edge_attr_dev)
        pred = pred_scaled * scale
        
        mb = m.expand(pred.shape[0], -1)
        pe = pred[mb]
        ye = yb[mb]
        ae.append(torch.abs(pe - ye).detach().cpu())
        se.append(((pe - ye) ** 2).detach().cpu())
        
    ae_all = torch.cat(ae)
    se_all = torch.cat(se)
    mae = float(ae_all.mean().item())
    rmse = float(torch.sqrt(se_all.mean()).item())
    return mae, rmse

MAX_EPOCHS = 20
PATIENCE = 4

best_val = float('inf')
best_epoch = -1
best_state = None
no_improve = 0
history = []

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    lam = lambda_phys_schedule(epoch - 1)

    tr_total, tr_data, tr_phys = [], [], []
    for xb, yb_scaled in train_loader:
        stats = train_step_amp(
            model=model,
            optimizer=optimizer,
            x_seq=xb,
            target=yb_scaled,
            edge_index=edge_index_dev,
            edge_attr=edge_attr_dev,
            train_mask=train_mask_dev,
            upwind_edge_mask=upwind_mask_dev,
            device=device,
            scaler=scaler_amp,
            physics_lambda=float(lam),
        )
        tr_total.append(stats['loss_total'])
        tr_data.append(stats['loss_data'])
        tr_phys.append(stats['loss_physics'])

    val_data_scaled = eval_masked_loss_scaled(model, val_loader)

    row = {
        'epoch': epoch,
        'lambda_phys': float(lam),
        'train_total': float(np.mean(tr_total)),
        'train_data': float(np.mean(tr_data)),
        'train_phys': float(np.mean(tr_phys)),
        'val_data_scaled_mse': float(val_data_scaled),
    }
    history.append(row)

    print(
        f"Epoch {epoch:02d} | lambda_phys={row['lambda_phys']:.4f} | "
        f"train_total={row['train_total']:.6f} | train_data={row['train_data']:.6f} | "
        f"train_phys={row['train_phys']:.6f} | val_data_scaled_mse={row['val_data_scaled_mse']:.6f}"
    )

    if row['val_data_scaled_mse'] < best_val - 1e-6:
        best_val = row['val_data_scaled_mse']
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        no_improve = 0
    else:
        no_improve += 1

    if no_improve >= PATIENCE:
        print(f"Early stopping at epoch {epoch}. Best epoch: {best_epoch}, best_scaled_mse={best_val:.6f}")
        break

if best_state is not None:
    model.load_state_dict(best_state)

mae_u, rmse_u = eval_masked_mae_rmse_unscaled(model, val_loader, TARGET_SCALE)

print({
    'best_epoch': int(best_epoch),
    'best_val_scaled_mse': float(best_val),
    'val_mae_unscaled': float(mae_u),
    'val_rmse_unscaled': float(rmse_u),
    'target_scale_q95': float(TARGET_SCALE),
    'physics_enabled': bool(USE_PHYSICS)
})

Epoch 01 | lambda_phys=0.0312 | train_total=0.066639 | train_data=0.065797 | train_phys=0.026956 | val_data_scaled_mse=0.006566
Epoch 02 | lambda_phys=0.0625 | train_total=0.033404 | train_data=0.032747 | train_phys=0.010523 | val_data_scaled_mse=0.003621
Epoch 03 | lambda_phys=0.0938 | train_total=0.024541 | train_data=0.024031 | train_phys=0.005445 | val_data_scaled_mse=0.004127
Epoch 04 | lambda_phys=0.1250 | train_total=0.019816 | train_data=0.019398 | train_phys=0.003343 | val_data_scaled_mse=0.002754
Epoch 05 | lambda_phys=0.1562 | train_total=0.016534 | train_data=0.016202 | train_phys=0.002130 | val_data_scaled_mse=0.002702
Epoch 06 | lambda_phys=0.1875 | train_total=0.014773 | train_data=0.014536 | train_phys=0.001267 | val_data_scaled_mse=0.002311
Epoch 07 | lambda_phys=0.2188 | train_total=0.013634 | train_data=0.013439 | train_phys=0.000891 | val_data_scaled_mse=0.002320
Epoch 08 | lambda_phys=0.2500 | train_total=0.013255 | train_data=0.013070 | train_phys=0.000739 | val_d

In [27]:
phase6_result = {
    'best_val_scaled_mse': 0.0013362345814874226,
    'val_mae_unscaled': 6.013996601104736,
    'val_rmse_unscaled': 8.214521408081055
}

In [25]:
# PHASE 6B - Retune physics schedule and compare against previous run
import copy
import numpy as np
import torch

# Keep a record of the previous run if present for delta analysis
prev_result = globals().get('phase6_result', None)

# Reinitialize model/optimizer for fair retune
model = STPIGNN(
    node_in_dim=int(x_time_node_feat.shape[-1]),
    edge_dim=int(edge_attr.shape[-1]),
    spatial_hidden_dim=96,
    temporal_hidden_dim=96,
    gnn_layers=2,
    gru_layers=1,
    dropout=0.15,
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=5e-4)
scaler_amp = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

if not ('upwind_mask_dev' in globals()) or upwind_mask_dev is None:
    raise RuntimeError('upwind_mask_dev is missing. Run physics mask build cell first.')

# Longer warmup allows the GNN layers to learn spatial structure before physics constraints kick in
def lambda_phys_schedule(epoch, warmup_epochs=10, max_lambda=0.15):
    if epoch < warmup_epochs:
        return max_lambda * float(epoch + 1) / float(warmup_epochs)
    return max_lambda

@torch.no_grad()
def eval_masked_loss_scaled(model, loader):
    model.eval()
    vals = []
    for xb, yb_scaled in loader:
        xb = xb.to(device, non_blocking=True)
        yb_scaled = yb_scaled.to(device, non_blocking=True)
        pred_scaled = model(x_seq=xb, edge_index=edge_index_dev, edge_attr=edge_attr_dev)
        loss = masked_data_loss(pred=pred_scaled, target=yb_scaled, train_mask=train_mask_dev)
        vals.append(float(loss.detach().cpu().item()))
    return float(np.mean(vals)) if vals else float('nan')

@torch.no_grad()
def eval_masked_mae_rmse_unscaled(model, loader, scale):
    model.eval()
    m = train_mask_dev.view(1, -1)
    ae, se = [], []
    for xb, yb_scaled in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb_scaled.to(device, non_blocking=True) * scale
        pred_scaled = model(x_seq=xb, edge_index=edge_index_dev, edge_attr=edge_attr_dev)
        pred = pred_scaled * scale
        mb = m.expand(pred.shape[0], -1)
        pe = pred[mb]
        ye = yb[mb]
        ae.append(torch.abs(pe - ye).detach().cpu())
        se.append(((pe - ye) ** 2).detach().cpu())
    ae_all = torch.cat(ae)
    se_all = torch.cat(se)
    return float(ae_all.mean().item()), float(torch.sqrt(se_all.mean()).item())

MAX_EPOCHS = 20
PATIENCE = 4

best_val = float('inf')
best_epoch = -1
best_state = None
no_improve = 0
history_6b = []

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    # Apply the retuned schedule
    lam = lambda_phys_schedule(epoch - 1, warmup_epochs=10, max_lambda=0.15)

    tr_total, tr_data, tr_phys = [], [], []
    for xb, yb_scaled in train_loader:
        stats = train_step_amp(
            model=model,
            optimizer=optimizer,
            x_seq=xb,
            target=yb_scaled,
            edge_index=edge_index_dev,
            edge_attr=edge_attr_dev,
            train_mask=train_mask_dev,
            upwind_edge_mask=upwind_mask_dev,
            device=device,
            scaler=scaler_amp,
            physics_lambda=float(lam),
        )
        tr_total.append(stats['loss_total'])
        tr_data.append(stats['loss_data'])
        tr_phys.append(stats['loss_physics'])

    val_data_scaled = eval_masked_loss_scaled(model, val_loader)
    row = {
        'epoch': epoch,
        'lambda_phys': float(lam),
        'train_total': float(np.mean(tr_total)),
        'train_data': float(np.mean(tr_data)),
        'train_phys': float(np.mean(tr_phys)),
        'val_data_scaled_mse': float(val_data_scaled),
    }
    history_6b.append(row)

    print(
        f"Epoch {epoch:02d} | lambda_phys={row['lambda_phys']:.4f} | "
        f"train_total={row['train_total']:.6f} | train_data={row['train_data']:.6f} | "
        f"train_phys={row['train_phys']:.6f} | val_data_scaled_mse={row['val_data_scaled_mse']:.6f}"
    )

    if row['val_data_scaled_mse'] < best_val - 1e-6:
        best_val = row['val_data_scaled_mse']
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        no_improve = 0
    else:
        no_improve += 1

    if no_improve >= PATIENCE:
        print(f"Early stopping at epoch {epoch}. Best epoch: {best_epoch}, best_scaled_mse={best_val:.6f}")
        break

if best_state is not None:
    model.load_state_dict(best_state)

mae_u, rmse_u = eval_masked_mae_rmse_unscaled(model, val_loader, TARGET_SCALE)

phase6b_result = {
    'best_epoch': int(best_epoch),
    'best_val_scaled_mse': float(best_val),
    'val_mae_unscaled': float(mae_u),
    'val_rmse_unscaled': float(rmse_u),
    'target_scale_q95': float(TARGET_SCALE),
    'physics_enabled': True,
    'lambda_max': 0.15,
    'warmup_epochs': 10,
}

print('\n' + '='*30)
print('PHASE 6B RESULT')
print(phase6b_result)

if prev_result is not None:
    print('\nPREVIOUS RESULT')
    print(prev_result)
    print('\nDELTA (6B - previous)')
    print({
        'delta_scaled_mse': phase6b_result['best_val_scaled_mse'] - prev_result.get('best_val_scaled_mse', np.nan),
        'delta_mae': phase6b_result['val_mae_unscaled'] - prev_result.get('val_mae_unscaled', np.nan),
        'delta_rmse': phase6b_result['val_rmse_unscaled'] - prev_result.get('val_rmse_unscaled', np.nan),
    })

Epoch 01 | lambda_phys=0.0150 | train_total=0.057039 | train_data=0.056700 | train_phys=0.022593 | val_data_scaled_mse=0.008421
Epoch 02 | lambda_phys=0.0300 | train_total=0.024066 | train_data=0.023741 | train_phys=0.010828 | val_data_scaled_mse=0.003334
Epoch 03 | lambda_phys=0.0450 | train_total=0.017621 | train_data=0.017384 | train_phys=0.005260 | val_data_scaled_mse=0.002365
Epoch 04 | lambda_phys=0.0600 | train_total=0.014281 | train_data=0.014091 | train_phys=0.003178 | val_data_scaled_mse=0.002155
Epoch 05 | lambda_phys=0.0750 | train_total=0.012869 | train_data=0.012701 | train_phys=0.002233 | val_data_scaled_mse=0.002008
Epoch 06 | lambda_phys=0.0900 | train_total=0.012281 | train_data=0.012126 | train_phys=0.001720 | val_data_scaled_mse=0.001871
Epoch 07 | lambda_phys=0.1050 | train_total=0.012035 | train_data=0.011886 | train_phys=0.001418 | val_data_scaled_mse=0.005537
Epoch 08 | lambda_phys=0.1200 | train_total=0.013201 | train_data=0.013040 | train_phys=0.001340 | val_d

In [28]:
# PHASE 7 - Save best model and comparison report
import pandas as pd
import torch
from pathlib import Path

ckpt_dir = root / 'data/gnn/checkpoints'
ckpt_dir.mkdir(parents=True, exist_ok=True)

final_ckpt = ckpt_dir / 'stpignn_local_physics_v2_best.pt'
history_csv = ckpt_dir / 'stpignn_local_physics_v2_history.csv'
compare_csv = ckpt_dir / 'stpignn_local_comparison.csv'

# Consolidating all training metadata into the checkpoint for reproducibility
torch.save({
    'model_state_dict': model.state_dict(),
    'best_epoch': int(phase6b_result['best_epoch']),
    'best_val_scaled_mse': float(phase6b_result['best_val_scaled_mse']),
    'val_mae_unscaled': float(phase6b_result['val_mae_unscaled']),
    'val_rmse_unscaled': float(phase6b_result['val_rmse_unscaled']),
    'target_scale_q95': float(phase6b_result['target_scale_q95']),
    'physics_enabled': bool(phase6b_result['physics_enabled']),
    'lambda_max': float(phase6b_result['lambda_max']),
    'warmup_epochs': int(phase6b_result['warmup_epochs']),
    'window': int(WINDOW),
    'num_nodes_train': int(num_nodes),
    'num_edges_train': int(edge_index.shape[1]),
    'sensor_nodes_train': int(train_mask.sum().item()),
}, final_ckpt)

pd.DataFrame(history_6b).to_csv(history_csv, index=False)

rows = []
if 'phase6_result' in globals() and phase6_result is not None:
    rows.append({
        'run': 'previous',
        'best_val_scaled_mse': float(phase6_result.get('best_val_scaled_mse', float('nan'))),
        'val_mae_unscaled': float(phase6_result.get('val_mae_unscaled', float('nan'))),
        'val_rmse_unscaled': float(phase6_result.get('val_rmse_unscaled', float('nan'))),
    })

rows.append({
    'run': 'phase6b_best',
    'best_val_scaled_mse': float(phase6b_result['best_val_scaled_mse']),
    'val_mae_unscaled': float(phase6b_result['val_mae_unscaled']),
    'val_rmse_unscaled': float(phase6b_result['val_rmse_unscaled']),
})

comp = pd.DataFrame(rows)
comp.to_csv(compare_csv, index=False)

print('Saved checkpoint:', final_ckpt)
print('Saved history:', history_csv)
print('Saved comparison:', compare_csv)
print('\nFinal Comparison Summary:')
print(comp)

Saved checkpoint: /home/jupyter-1nt23cb058/Capstone/data/gnn/checkpoints/stpignn_local_physics_v2_best.pt
Saved history: /home/jupyter-1nt23cb058/Capstone/data/gnn/checkpoints/stpignn_local_physics_v2_history.csv
Saved comparison: /home/jupyter-1nt23cb058/Capstone/data/gnn/checkpoints/stpignn_local_comparison.csv

Final Comparison Summary:
            run  best_val_scaled_mse  val_mae_unscaled  val_rmse_unscaled
0      previous             0.001336          6.013997           8.214521
1  phase6b_best             0.001141          5.633329           7.542551


In [29]:
# PHASE 8 - Temporal blocked folds on local subgraph (validation-grade)
import copy
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

required = [
    'x_time_node_feat', 'y_time_node', 'train_mask', 'edge_index_dev', 'edge_attr_dev',
    'train_mask_dev', 'upwind_mask_dev', 'TARGET_SCALE', 'WINDOW', 'device'
]
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError(f'Missing prerequisites for Phase 8: {missing}')

class WindowDataset(Dataset):
    def __init__(self, x_tnf, y_tn_scaled, start_t, end_t, window):
        self.x = x_tnf
        self.y = y_tn_scaled
        self.window = window
        self.indices = list(range(start_t, end_t - window))
        if len(self.indices) <= 0:
            raise ValueError('No windows available for this fold split')

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        t0 = self.indices[i]
        return self.x[t0:t0 + self.window], self.y[t0 + self.window]

def lambda_phys_schedule(epoch, warmup_epochs=10, max_lambda=0.15):
    if upwind_mask_dev is None:
        return 0.0
    if epoch < warmup_epochs:
        return max_lambda * float(epoch + 1) / float(warmup_epochs)
    return max_lambda

@torch.no_grad()
def eval_masked_loss_scaled(model, loader):
    model.eval()
    vals = []
    for xb, yb_scaled in loader:
        xb = xb.to(device, non_blocking=True)
        yb_scaled = yb_scaled.to(device, non_blocking=True)
        pred_scaled = model(x_seq=xb, edge_index=edge_index_dev, edge_attr=edge_attr_dev)
        loss = masked_data_loss(pred=pred_scaled, target=yb_scaled, train_mask=train_mask_dev)
        vals.append(float(loss.detach().cpu().item()))
    return float(np.mean(vals)) if vals else float('nan')

@torch.no_grad()
def eval_masked_mae_rmse_unscaled(model, loader, scale):
    model.eval()
    m = train_mask_dev.view(1, -1)
    ae, se = [], []
    for xb, yb_scaled in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb_scaled.to(device, non_blocking=True) * scale
        pred = model(x_seq=xb, edge_index=edge_index_dev, edge_attr=edge_attr_dev) * scale
        mb = m.expand(pred.shape[0], -1)
        pe = pred[mb]
        ye = yb[mb]
        ae.append(torch.abs(pe - ye).detach().cpu())
        se.append(((pe - ye) ** 2).detach().cpu())
    ae_all = torch.cat(ae)
    se_all = torch.cat(se)
    return float(ae_all.mean().item()), float(torch.sqrt(se_all.mean()).item())

x_all = x_time_node_feat
y_all_scaled = y_time_node / TARGET_SCALE
T_total = x_all.shape[0]

# Fold plan: rolling blocked validation windows from the end
N_FOLDS = 3
VAL_HOURS = 14 * 24  # 2 weeks per validation block
MIN_TRAIN_HOURS = 45 * 24  # At least 45 days of training history

fold_ranges = []
end_t = T_total
for i in range(N_FOLDS):
    val_end = end_t - i * VAL_HOURS
    val_start = val_end - VAL_HOURS
    train_end = val_start
    train_start = 0
    if train_end - train_start < MIN_TRAIN_HOURS:
        continue
    if val_start < 0 or val_end > T_total:
        continue
    fold_ranges.append((train_start, train_end, val_start, val_end))

fold_ranges = list(reversed(fold_ranges))
if len(fold_ranges) == 0:
    raise RuntimeError('No valid temporal folds could be created with current settings')

print('Fold ranges (train_start, train_end, val_start, val_end):')
for i, r in enumerate(fold_ranges, 1):
    print(f'Fold {i}: {r}')

fold_results = []

for fold_id, (tr_s, tr_e, va_s, va_e) in enumerate(fold_ranges, 1):
    print(f'\n--- Starting Fold {fold_id} ---')
    train_ds_f = WindowDataset(x_all, y_all_scaled, start_t=tr_s, end_t=tr_e, window=WINDOW)
    val_ds_f = WindowDataset(x_all, y_all_scaled, start_t=va_s - WINDOW, end_t=va_e, window=WINDOW)

    train_loader_f = DataLoader(train_ds_f, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
    val_loader_f = DataLoader(val_ds_f, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

    model_f = STPIGNN(
        node_in_dim=int(x_all.shape[-1]),
        edge_dim=int(edge_attr_dev.shape[-1]),
        spatial_hidden_dim=96,
        temporal_hidden_dim=96,
        gnn_layers=2,
        gru_layers=1,
        dropout=0.15,
    ).to(device)

    opt_f = torch.optim.AdamW(model_f.parameters(), lr=3e-4, weight_decay=5e-4)
    scaler_f = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

    MAX_EPOCHS = 15
    PATIENCE = 3
    best_val = float('inf')
    best_epoch = -1
    best_state = None
    no_improve = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        model_f.train()
        lam = lambda_phys_schedule(epoch - 1, warmup_epochs=10, max_lambda=0.15)

        for xb, yb_scaled in train_loader_f:
            from gnn.model import train_step_amp # Ensure visibility
            _ = train_step_amp(
                model=model_f,
                optimizer=opt_f,
                x_seq=xb,
                target=yb_scaled,
                edge_index=edge_index_dev,
                edge_attr=edge_attr_dev,
                train_mask=train_mask_dev,
                upwind_edge_mask=upwind_mask_dev,
                device=device,
                scaler=scaler_f,
                physics_lambda=float(lam),
            )

        val_scaled = eval_masked_loss_scaled(model_f, val_loader_f)
        if val_scaled < best_val - 1e-6:
            best_val = val_scaled
            best_epoch = epoch
            best_state = copy.deepcopy(model_f.state_dict())
            no_improve = 0
        else:
            no_improve += 1
        
        if no_improve >= PATIENCE:
            break

    if best_state is not None:
        model_f.load_state_dict(best_state)

    mae_u, rmse_u = eval_masked_mae_rmse_unscaled(model_f, val_loader_f, TARGET_SCALE)

    fold_result = {
        'fold': int(fold_id),
        'train_hours': int(tr_e - tr_s),
        'val_hours': int(va_e - va_s),
        'best_epoch': int(best_epoch),
        'best_val_scaled_mse': float(best_val),
        'val_mae_unscaled': float(mae_u),
        'val_rmse_unscaled': float(rmse_u),
    }
    fold_results.append(fold_result)
    print(f'Fold {fold_id} result:', fold_result)

fold_df = pd.DataFrame(fold_results)
summary = {
    'folds': int(len(fold_df)),
    'avg_best_val_scaled_mse': float(fold_df['best_val_scaled_mse'].mean()),
    'avg_val_mae_unscaled': float(fold_df['val_mae_unscaled'].mean()),
    'avg_val_rmse_unscaled': float(fold_df['val_rmse_unscaled'].mean()),
}

print('\n' + '='*30)
print('CV Summary:', summary)

out_dir = root / 'data/gnn/checkpoints'
out_dir.mkdir(parents=True, exist_ok=True)
fold_csv = out_dir / 'stpignn_local_temporal_cv_folds.csv'
summary_csv = out_dir / 'stpignn_local_temporal_cv_summary.csv'
fold_df.to_csv(fold_csv, index=False)
pd.DataFrame([summary]).to_csv(summary_csv, index=False)

print('Saved fold metrics:', fold_csv)
print('Saved summary:', summary_csv)

Fold ranges (train_start, train_end, val_start, val_end):
Fold 1: (0, 1153, 1153, 1489)
Fold 2: (0, 1489, 1489, 1825)
Fold 3: (0, 1825, 1825, 2161)

--- Starting Fold 1 ---
Fold 1 result: {'fold': 1, 'train_hours': 1153, 'val_hours': 336, 'best_epoch': 15, 'best_val_scaled_mse': 0.0014303822355854901, 'val_mae_unscaled': 5.871687889099121, 'val_rmse_unscaled': 8.49609375}

--- Starting Fold 2 ---
Fold 2 result: {'fold': 2, 'train_hours': 1489, 'val_hours': 336, 'best_epoch': 8, 'best_val_scaled_mse': 0.0014013844248395237, 'val_mae_unscaled': 5.337104320526123, 'val_rmse_unscaled': 8.409534454345703}

--- Starting Fold 3 ---
Fold 3 result: {'fold': 3, 'train_hours': 1825, 'val_hours': 336, 'best_epoch': 6, 'best_val_scaled_mse': 0.0017716559315366404, 'val_mae_unscaled': 6.357393264770508, 'val_rmse_unscaled': 9.455466270446777}

CV Summary: {'folds': 3, 'avg_best_val_scaled_mse': 0.0015344741973205515, 'avg_val_mae_unscaled': 5.855395158131917, 'avg_val_rmse_unscaled': 8.7870314915974

In [30]:
# PHASE 9.1 - Load fold metrics and compute stability stats
import pandas as pd
import numpy as np
from pathlib import Path

ckpt_dir = root / 'data/gnn/checkpoints'
fold_csv = ckpt_dir / 'stpignn_local_temporal_cv_folds.csv'

fold_df = pd.read_csv(fold_csv)
print("Fold-by-Fold Performance:\n", fold_df)

summary_robust = {
    'folds': int(len(fold_df)),
    'mean_rmse': float(fold_df['val_rmse_unscaled'].mean()),
    'std_rmse': float(fold_df['val_rmse_unscaled'].std(ddof=1)),
    'worst_rmse': float(fold_df['val_rmse_unscaled'].max()),
    'mean_mae': float(fold_df['val_mae_unscaled'].mean()),
    'std_mae': float(fold_df['val_mae_unscaled'].std(ddof=1)),
    'worst_mae': float(fold_df['val_mae_unscaled'].max()),
    'mean_scaled_mse': float(fold_df['best_val_scaled_mse'].mean()),
    'std_scaled_mse': float(fold_df['best_val_scaled_mse'].std(ddof=1)),
}

print('\nRobust summary:', summary_robust)

# Conservative config choice based on fold variability to ensure generalization
selected_cfg = {
    'lambda_max': 0.12,
    'warmup_epochs': 12,
    'max_epochs': 20,
    'patience': 4,
    'batch_size': 8
}
print('Selected robust config:', selected_cfg)


# PHASE 9.2 - Build full-window dataset for final local retrain
from torch.utils.data import Dataset, DataLoader
import torch

class WindowDataset(Dataset):
    def __init__(self, x_tnf, y_tn_scaled, start_t, end_t, window):
        self.x = x_tnf
        self.y = y_tn_scaled
        self.window = window
        self.indices = list(range(start_t, end_t - window))
        if len(self.indices) <= 0:
            raise ValueError('No windows available')

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        t0 = self.indices[i]
        return self.x[t0:t0 + self.window], self.y[t0 + self.window]

y_all_scaled = y_time_node / TARGET_SCALE
full_train_ds = WindowDataset(x_time_node_feat, y_all_scaled, start_t=0, end_t=x_time_node_feat.shape[0], window=WINDOW)
full_train_loader = DataLoader(
    full_train_ds,
    batch_size=int(selected_cfg['batch_size']),
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

print({'full_windows': len(full_train_ds), 'window': int(WINDOW)})


# PHASE 9.3 - Retrain final robust local model on full local data
import copy
from gnn.model import STPIGNN, train_step_amp

final_model = STPIGNN(
    node_in_dim=int(x_time_node_feat.shape[-1]),
    edge_dim=int(edge_attr_dev.shape[-1]),
    spatial_hidden_dim=96,
    temporal_hidden_dim=96,
    gnn_layers=2,
    gru_layers=1,
    dropout=0.15,
).to(device)

final_opt = torch.optim.AdamW(final_model.parameters(), lr=3e-4, weight_decay=5e-4)
final_scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

def lambda_phys_schedule(epoch, warmup_epochs, max_lambda):
    if upwind_mask_dev is None:
        return 0.0
    if epoch < warmup_epochs:
        return max_lambda * float(epoch + 1) / float(warmup_epochs)
    return max_lambda

final_history = []
best_state = None
best_proxy = float('inf')
no_improve = 0

# Proxy is train_data moving average because this is full-data retrain with no holdout.
for epoch in range(1, int(selected_cfg['max_epochs']) + 1):
    final_model.train()
    lam = lambda_phys_schedule(
        epoch - 1,
        warmup_epochs=int(selected_cfg['warmup_epochs']),
        max_lambda=float(selected_cfg['lambda_max'])
    )

    tr_total, tr_data, tr_phys = [], [], []
    for xb, yb_scaled in full_train_loader:
        stats = train_step_amp(
            model=final_model,
            optimizer=final_opt,
            x_seq=xb,
            target=yb_scaled,
            edge_index=edge_index_dev,
            edge_attr=edge_attr_dev,
            train_mask=train_mask_dev,
            upwind_edge_mask=upwind_mask_dev,
            device=device,
            scaler=final_scaler,
            physics_lambda=float(lam),
        )
        tr_total.append(stats['loss_total'])
        tr_data.append(stats['loss_data'])
        tr_phys.append(stats['loss_physics'])

    row = {
        'epoch': epoch,
        'lambda_phys': float(lam),
        'train_total': float(np.mean(tr_total)),
        'train_data': float(np.mean(tr_data)),
        'train_phys': float(np.mean(tr_phys)),
    }
    final_history.append(row)

    print(
        f"Epoch {epoch:02d} | lambda_phys={row['lambda_phys']:.4f} | "
        f"train_total={row['train_total']:.6f} | train_data={row['train_data']:.6f} | "
        f"train_phys={row['train_phys']:.6f}"
    )

    proxy = row['train_data']
    if proxy < best_proxy - 1e-6:
        best_proxy = proxy
        best_state = copy.deepcopy(final_model.state_dict())
        no_improve = 0
    else:
        no_improve += 1

    if no_improve >= int(selected_cfg['patience']):
        print(f"Early stop final retrain at epoch {epoch}. Best proxy={best_proxy:.6f}")
        break

if best_state is not None:
    final_model.load_state_dict(best_state)


# PHASE 9.4 - Save robust final artifact + report
final_ckpt = ckpt_dir / 'stpignn_local_physics_cv_robust_final.pt'
final_hist_csv = ckpt_dir / 'stpignn_local_physics_cv_robust_final_history.csv'
final_report_csv = ckpt_dir / 'stpignn_local_physics_cv_robust_report.csv'

torch.save({
    'model_state_dict': final_model.state_dict(),
    'target_scale_q95': float(TARGET_SCALE),
    'window': int(WINDOW),
    'num_nodes_train': int(num_nodes),
    'num_edges_train': int(edge_index.shape[1]),
    'sensor_nodes_train': int(train_mask.sum().item()),
    'physics_enabled': bool(upwind_mask_dev is not None),
    'selected_cfg': selected_cfg,
    'cv_summary': summary_robust,
}, final_ckpt)

pd.DataFrame(final_history).to_csv(final_hist_csv, index=False)
pd.DataFrame([{**summary_robust, **selected_cfg}]).to_csv(final_report_csv, index=False)

print('Saved final checkpoint:', final_ckpt)
print('Saved final history:', final_hist_csv)
print('Saved robust report:', final_report_csv)

Fold-by-Fold Performance:
    fold  train_hours  val_hours  best_epoch  best_val_scaled_mse  \
0     1         1153        336          15             0.001430   
1     2         1489        336           8             0.001401   
2     3         1825        336           6             0.001772   

   val_mae_unscaled  val_rmse_unscaled  
0          5.871688           8.496094  
1          5.337104           8.409534  
2          6.357393           9.455466  

Robust summary: {'folds': 3, 'mean_rmse': 8.787031491597494, 'std_rmse': 0.580497130137186, 'worst_rmse': 9.455466270446776, 'mean_mae': 5.855395158131917, 'std_mae': 0.5103395656311663, 'worst_mae': 6.357393264770508, 'mean_scaled_mse': 0.0015344741973205, 'std_scaled_mse': 0.00020591648681281277}
Selected robust config: {'lambda_max': 0.12, 'warmup_epochs': 12, 'max_epochs': 20, 'patience': 4, 'batch_size': 8}
{'full_windows': 2149, 'window': 12}
Epoch 01 | lambda_phys=0.0100 | train_total=0.030465 | train_data=0.030198 | train

In [31]:
# PHASE 10.1 - Build strict OOT split
import torch
from torch.utils.data import Dataset, DataLoader

class WindowDataset(Dataset):
    def __init__(self, x_tnf, y_tn_scaled, start_t, end_t, window):
        self.x = x_tnf
        self.y = y_tn_scaled
        self.window = window
        self.indices = list(range(start_t, end_t - window))
        if len(self.indices) <= 0:
            raise ValueError('No windows available for this range')

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        t0 = self.indices[i]
        return self.x[t0:t0 + self.window], self.y[t0 + self.window]

HOLDOUT_HOURS = 14 * 24  # Strict 2-week final holdout
T_total = x_time_node_feat.shape[0]
train_end = T_total - HOLDOUT_HOURS

if train_end <= WINDOW + 24:
    raise RuntimeError('Not enough history for strict OOT split')

y_all_scaled = y_time_node / TARGET_SCALE

train_ds_oot = WindowDataset(x_time_node_feat, y_all_scaled, start_t=0, end_t=train_end, window=WINDOW)
test_ds_oot = WindowDataset(x_time_node_feat, y_all_scaled, start_t=train_end - WINDOW, end_t=T_total, window=WINDOW)

train_loader_oot = DataLoader(train_ds_oot, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
test_loader_oot = DataLoader(test_ds_oot, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

print({
    'train_end_t': int(train_end),
    'holdout_hours': int(HOLDOUT_HOURS),
    'train_windows': int(len(train_ds_oot)),
    'test_windows': int(len(test_ds_oot))
})


# PHASE 10.2 - Train with robust config, evaluate on strict OOT
import copy
import numpy as np
from gnn.model import STPIGNN, train_step_amp, masked_data_loss

final_cfg = {'lambda_max': 0.12, 'warmup_epochs': 12, 'max_epochs': 20, 'patience': 4}

def lambda_phys_schedule(epoch, warmup_epochs=12, max_lambda=0.12):
    if upwind_mask_dev is None:
        return 0.0
    if epoch < warmup_epochs:
        return max_lambda * float(epoch + 1) / float(warmup_epochs)
    return max_lambda

@torch.no_grad()
def eval_masked_loss_scaled(model, loader):
    model.eval()
    vals = []
    for xb, yb_scaled in loader:
        xb = xb.to(device, non_blocking=True)
        yb_scaled = yb_scaled.to(device, non_blocking=True)
        pred_scaled = model(x_seq=xb, edge_index=edge_index_dev, edge_attr=edge_attr_dev)
        loss = masked_data_loss(pred=pred_scaled, target=yb_scaled, train_mask=train_mask_dev)
        vals.append(float(loss.detach().cpu().item()))
    return float(np.mean(vals)) if vals else float('nan')

@torch.no_grad()
def eval_masked_mae_rmse_unscaled(model, loader, scale):
    model.eval()
    m = train_mask_dev.view(1, -1)
    ae, se = [], []
    for xb, yb_scaled in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb_scaled.to(device, non_blocking=True) * scale
        pred = model(x_seq=xb, edge_index=edge_index_dev, edge_attr=edge_attr_dev) * scale
        mb = m.expand(pred.shape[0], -1)
        pe = pred[mb]
        ye = yb[mb]
        ae.append(torch.abs(pe - ye).detach().cpu())
        se.append(((pe - ye) ** 2).detach().cpu())
    ae_all = torch.cat(ae)
    se_all = torch.cat(se)
    return float(ae_all.mean().item()), float(torch.sqrt(se_all.mean()).item())

model_oot = STPIGNN(
    node_in_dim=int(x_time_node_feat.shape[-1]),
    edge_dim=int(edge_attr_dev.shape[-1]),
    spatial_hidden_dim=96,
    temporal_hidden_dim=96,
    gnn_layers=2,
    gru_layers=1,
    dropout=0.15,
).to(device)

opt_oot = torch.optim.AdamW(model_oot.parameters(), lr=3e-4, weight_decay=5e-4)
scaler_oot = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

best_state = None
best_val = float('inf')
best_epoch = -1
no_improve = 0

for epoch in range(1, int(final_cfg['max_epochs']) + 1):
    model_oot.train()
    lam = lambda_phys_schedule(epoch - 1, warmup_epochs=int(final_cfg['warmup_epochs']), max_lambda=float(final_cfg['lambda_max']))

    for xb, yb_scaled in train_loader_oot:
        _ = train_step_amp(
            model=model_oot,
            optimizer=opt_oot,
            x_seq=xb,
            target=yb_scaled,
            edge_index=edge_index_dev,
            edge_attr=edge_attr_dev,
            train_mask=train_mask_dev,
            upwind_edge_mask=upwind_mask_dev,
            device=device,
            scaler=scaler_oot,
            physics_lambda=float(lam),
        )

    val_scaled = eval_masked_loss_scaled(model_oot, test_loader_oot)
    print(f"Epoch {epoch:02d} | lambda_phys={lam:.4f} | oot_scaled_mse={val_scaled:.6f}")

    if val_scaled < best_val - 1e-6:
        best_val = val_scaled
        best_epoch = epoch
        best_state = copy.deepcopy(model_oot.state_dict())
        no_improve = 0
    else:
        no_improve += 1

    if no_improve >= int(final_cfg['patience']):
        print(f"Early stop at epoch {epoch}; best_epoch={best_epoch}, best_oot_scaled_mse={best_val:.6f}")
        break

if best_state is not None:
    model_oot.load_state_dict(best_state)

oot_mae, oot_rmse = eval_masked_mae_rmse_unscaled(model_oot, test_loader_oot, TARGET_SCALE)
oot_result = {
    'best_epoch': int(best_epoch),
    'best_oot_scaled_mse': float(best_val),
    'oot_mae_unscaled': float(oot_mae),
    'oot_rmse_unscaled': float(oot_rmse),
    'target_scale_q95': float(TARGET_SCALE)
}
print('\n' + '='*30)
print('STRICT OOT RESULT:', oot_result)


# PHASE 10.3 - Save strict OOT result
import pandas as pd

ckpt_dir = root / 'data/gnn/checkpoints'
ckpt_dir.mkdir(parents=True, exist_ok=True)

oot_ckpt = ckpt_dir / 'stpignn_local_physics_strict_oot_best.pt'
oot_csv = ckpt_dir / 'stpignn_local_physics_strict_oot_result.csv'

torch.save({
    'model_state_dict': model_oot.state_dict(),
    'oot_result': oot_result,
    'final_cfg': final_cfg,
    'window': int(WINDOW),
    'sensor_nodes_train': int(train_mask.sum().item()),
}, oot_ckpt)

pd.DataFrame([oot_result]).to_csv(oot_csv, index=False)

print('Saved strict OOT checkpoint:', oot_ckpt)
print('Saved strict OOT metrics:', oot_csv)

{'train_end_t': 1825, 'holdout_hours': 336, 'train_windows': 1813, 'test_windows': 336}
Epoch 01 | lambda_phys=0.0100 | oot_scaled_mse=0.006461
Epoch 02 | lambda_phys=0.0200 | oot_scaled_mse=0.003226
Epoch 03 | lambda_phys=0.0300 | oot_scaled_mse=0.002891
Epoch 04 | lambda_phys=0.0400 | oot_scaled_mse=0.002491
Epoch 05 | lambda_phys=0.0500 | oot_scaled_mse=0.002586
Epoch 06 | lambda_phys=0.0600 | oot_scaled_mse=0.002447
Epoch 07 | lambda_phys=0.0700 | oot_scaled_mse=0.002246
Epoch 08 | lambda_phys=0.0800 | oot_scaled_mse=0.002027
Epoch 09 | lambda_phys=0.0900 | oot_scaled_mse=0.001786
Epoch 10 | lambda_phys=0.1000 | oot_scaled_mse=0.001949
Epoch 11 | lambda_phys=0.1100 | oot_scaled_mse=0.001988
Epoch 12 | lambda_phys=0.1200 | oot_scaled_mse=0.002562
Epoch 13 | lambda_phys=0.1200 | oot_scaled_mse=0.002730
Early stop at epoch 13; best_epoch=9, best_oot_scaled_mse=0.001786

STRICT OOT RESULT: {'best_epoch': 9, 'best_oot_scaled_mse': 0.0017857158672995865, 'oot_mae_unscaled': 5.98694467544

In [32]:
# PHASE 10B.1 - Build strict train/val/test windows
import torch
from torch.utils.data import Dataset, DataLoader

class WindowDataset(Dataset):
    def __init__(self, x_tnf, y_tn_scaled, start_t, end_t, window):
        self.x = x_tnf
        self.y = y_tn_scaled
        self.window = window
        self.indices = list(range(start_t, end_t - window))
        if len(self.indices) <= 0:
            raise ValueError('No windows for this split')

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        t0 = self.indices[i]
        return self.x[t0:t0 + self.window], self.y[t0 + self.window]

HOLDOUT_HOURS = 14 * 24 # 2 weeks for final unbiased test
VAL_HOURS = 14 * 24     # 2 weeks for unbiased early-stopping
T_total = x_time_node_feat.shape[0]

test_start = T_total - HOLDOUT_HOURS
val_start = test_start - VAL_HOURS
train_end = val_start

if train_end <= WINDOW + 24:
    raise RuntimeError('Not enough history for train/val/test split')

y_all_scaled = y_time_node / TARGET_SCALE

train_ds = WindowDataset(x_time_node_feat, y_all_scaled, 0, train_end, WINDOW)
# Validation set starts right after training
val_ds = WindowDataset(x_time_node_feat, y_all_scaled, val_start - WINDOW, test_start, WINDOW)
# Test set is the final untouched horizon
test_ds = WindowDataset(x_time_node_feat, y_all_scaled, test_start - WINDOW, T_total, WINDOW)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

print({
    'train_end_t': int(train_end),
    'val_start_t': int(val_start),
    'test_start_t': int(test_start),
    'train_windows': int(len(train_ds)),
    'val_windows': int(len(val_ds)),
    'test_windows': int(len(test_ds))
})

# PHASE 10B.2 - Train with early stop on validation only, then evaluate test once
import copy
import numpy as np
from gnn.model import STPIGNN, train_step_amp, masked_data_loss

def lambda_phys_schedule(epoch, warmup_epochs=12, max_lambda=0.12):
    if upwind_mask_dev is None:
        return 0.0
    if epoch < warmup_epochs:
        return max_lambda * float(epoch + 1) / float(warmup_epochs)
    return max_lambda

@torch.no_grad()
def eval_masked_loss_scaled(model, loader):
    model.eval()
    vals = []
    for xb, yb_scaled in loader:
        xb = xb.to(device, non_blocking=True)
        yb_scaled = yb_scaled.to(device, non_blocking=True)
        pred_scaled = model(x_seq=xb, edge_index=edge_index_dev, edge_attr=edge_attr_dev)
        loss = masked_data_loss(pred=pred_scaled, target=yb_scaled, train_mask=train_mask_dev)
        vals.append(float(loss.detach().cpu().item()))
    return float(np.mean(vals)) if vals else float('nan')

@torch.no_grad()
def eval_masked_mae_rmse_unscaled(model, loader, scale):
    model.eval()
    m = train_mask_dev.view(1, -1)
    ae, se = [], []
    for xb, yb_scaled in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb_scaled.to(device, non_blocking=True) * scale
        pred = model(x_seq=xb, edge_index=edge_index_dev, edge_attr=edge_attr_dev) * scale
        mb = m.expand(pred.shape[0], -1)
        pe = pred[mb]
        ye = yb[mb]
        ae.append(torch.abs(pe - ye).detach().cpu())
        se.append(((pe - ye) ** 2).detach().cpu())
    ae_all = torch.cat(ae)
    se_all = torch.cat(se)
    return float(ae_all.mean().item()), float(torch.sqrt(se_all.mean()).item())

model = STPIGNN(
    node_in_dim=int(x_time_node_feat.shape[-1]),
    edge_dim=int(edge_attr_dev.shape[-1]),
    spatial_hidden_dim=96,
    temporal_hidden_dim=96,
    gnn_layers=2,
    gru_layers=1,
    dropout=0.15,
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=5e-4)
scaler_amp = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

MAX_EPOCHS = 20
PATIENCE = 4

best_val = float('inf')
best_epoch = -1
best_state = None
no_improve = 0

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    lam = lambda_phys_schedule(epoch - 1, warmup_epochs=12, max_lambda=0.12)

    for xb, yb_scaled in train_loader:
        _ = train_step_amp(
            model=model,
            optimizer=optimizer,
            x_seq=xb,
            target=yb_scaled,
            edge_index=edge_index_dev,
            edge_attr=edge_attr_dev,
            train_mask=train_mask_dev,
            upwind_edge_mask=upwind_mask_dev,
            device=device,
            scaler=scaler_amp,
            physics_lambda=float(lam),
        )

    # Early stopping only looks at the validation block
    val_scaled = eval_masked_loss_scaled(model, val_loader)
    print(f"Epoch {epoch:02d} | lambda_phys={lam:.4f} | val_scaled_mse={val_scaled:.6f}")

    if val_scaled < best_val - 1e-6:
        best_val = val_scaled
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        no_improve = 0
    else:
        no_improve += 1

    if no_improve >= PATIENCE:
        print(f"Early stop at epoch {epoch}; best_epoch={best_epoch}, best_val_scaled_mse={best_val:.6f}")
        break

if best_state is not None:
    model.load_state_dict(best_state)

# Evaluate on the untouched test block once
test_scaled = eval_masked_loss_scaled(model, test_loader)
test_mae, test_rmse = eval_masked_mae_rmse_unscaled(model, test_loader, TARGET_SCALE)

phase10b_result = {
    'best_epoch_by_val': int(best_epoch),
    'best_val_scaled_mse': float(best_val),
    'test_scaled_mse': float(test_scaled),
    'test_mae_unscaled': float(test_mae),
    'test_rmse_unscaled': float(test_rmse),
}
print('\n' + '='*30)
print('PHASE 10B RESULT (UNBIASED TEST):', phase10b_result)

{'train_end_t': 1489, 'val_start_t': 1489, 'test_start_t': 1825, 'train_windows': 1477, 'val_windows': 336, 'test_windows': 336}
Epoch 01 | lambda_phys=0.0100 | val_scaled_mse=0.009770
Epoch 02 | lambda_phys=0.0200 | val_scaled_mse=0.004014
Epoch 03 | lambda_phys=0.0300 | val_scaled_mse=0.002758
Epoch 04 | lambda_phys=0.0400 | val_scaled_mse=0.002843
Epoch 05 | lambda_phys=0.0500 | val_scaled_mse=0.002341
Epoch 06 | lambda_phys=0.0600 | val_scaled_mse=0.002647
Epoch 07 | lambda_phys=0.0700 | val_scaled_mse=0.002119
Epoch 08 | lambda_phys=0.0800 | val_scaled_mse=0.002385
Epoch 09 | lambda_phys=0.0900 | val_scaled_mse=0.002198
Epoch 10 | lambda_phys=0.1000 | val_scaled_mse=0.001498
Epoch 11 | lambda_phys=0.1100 | val_scaled_mse=0.001508
Epoch 12 | lambda_phys=0.1200 | val_scaled_mse=0.001197
Epoch 13 | lambda_phys=0.1200 | val_scaled_mse=0.001258
Epoch 14 | lambda_phys=0.1200 | val_scaled_mse=0.001362
Epoch 15 | lambda_phys=0.1200 | val_scaled_mse=0.000917
Epoch 16 | lambda_phys=0.1200 |

In [33]:
# PHASE 10B.3 - Test evaluation and save (run after training cell)

# Ensure best checkpoint is loaded
if best_state is not None:
    model.load_state_dict(best_state)

test_scaled = eval_masked_loss_scaled(model, test_loader)
test_mae, test_rmse = eval_masked_mae_rmse_unscaled(model, test_loader, TARGET_SCALE)

phase10b_result = {
    'best_epoch_by_val': int(best_epoch),
    'best_val_scaled_mse': float(best_val),
    'test_scaled_mse': float(test_scaled),
    'test_mae_unscaled': float(test_mae),
    'test_rmse_unscaled': float(test_rmse),
    'target_scale_q95': float(TARGET_SCALE),
}

print('PHASE 10B RESULT:', phase10b_result)

# Optional save
import pandas as pd
import torch
from pathlib import Path

ckpt_dir = root / 'data/gnn/checkpoints'
ckpt_dir.mkdir(parents=True, exist_ok=True)

out_ckpt = ckpt_dir / 'stpignn_local_physics_phase10b_best.pt'
out_csv = ckpt_dir / 'stpignn_local_physics_phase10b_result.csv'

torch.save({
    'model_state_dict': model.state_dict(),
    'phase10b_result': phase10b_result
}, out_ckpt)

pd.DataFrame([phase10b_result]).to_csv(out_csv, index=False)

print('Saved:', out_ckpt)
print('Saved:', out_csv)

PHASE 10B RESULT: {'best_epoch_by_val': 15, 'best_val_scaled_mse': 0.0009169053214247383, 'test_scaled_mse': 0.0009470369910732621, 'test_mae_unscaled': 5.080196380615234, 'test_rmse_unscaled': 6.913161277770996, 'target_scale_q95': 224.6432342529297}
Saved: /home/jupyter-1nt23cb058/Capstone/data/gnn/checkpoints/stpignn_local_physics_phase10b_best.pt
Saved: /home/jupyter-1nt23cb058/Capstone/data/gnn/checkpoints/stpignn_local_physics_phase10b_result.csv


In [34]:
# PHASE 10C - Build and save model card artifacts
import json
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

if 'phase10b_result' not in globals():
    raise RuntimeError('phase10b_result not found. Run Phase 10B evaluation first.')

ckpt_dir = root / 'data/gnn/checkpoints'
ckpt_dir.mkdir(parents=True, exist_ok=True)

model_card_json = ckpt_dir / 'stpignn_local_physics_phase10b_model_card.json'
model_card_md = ckpt_dir / 'stpignn_local_physics_phase10b_model_card.md'
model_card_csv = ckpt_dir / 'stpignn_local_physics_phase10b_model_card.csv'

# Extract Metrics
metrics = {
    'best_epoch_by_val': int(phase10b_result['best_epoch_by_val']),
    'best_val_scaled_mse': float(phase10b_result['best_val_scaled_mse']),
    'test_scaled_mse': float(phase10b_result['test_scaled_mse']),
    'test_mae_unscaled': float(phase10b_result['test_mae_unscaled']),
    'test_rmse_unscaled': float(phase10b_result['test_rmse_unscaled']),
    'target_scale_q95': float(phase10b_result['target_scale_q95']),
}

# Extract Configuration
config = {
    'architecture': 'STPIGNN (GINE + GRU)',
    'node_in_dim': int(x_time_node_feat.shape[-1]) if 'x_time_node_feat' in globals() else None,
    'edge_dim': int(edge_attr.shape[-1]) if 'edge_attr' in globals() else None,
    'spatial_hidden_dim': 96,
    'temporal_hidden_dim': 96,
    'gnn_layers': 2,
    'gru_layers': 1,
    'dropout': 0.15,
    'optimizer': 'AdamW',
    'learning_rate': 3e-4,
    'weight_decay': 5e-4,
    'window': int(WINDOW) if 'WINDOW' in globals() else None,
    'physics_enabled': bool('upwind_mask_dev' in globals() and upwind_mask_dev is not None),
    'physics_lambda_max': 0.12,
    'physics_warmup_epochs': 12,
}

# Extract Data Split Info
data_split = {
    'split_type': 'strict temporal train/val/test',
    'train_end_t': int(train_end) if 'train_end' in globals() else None,
    'val_start_t': int(val_start) if 'val_start' in globals() else None,
    'test_start_t': int(test_start) if 'test_start' in globals() else None,
    'train_windows': int(len(train_ds)) if 'train_ds' in globals() else None,
    'val_windows': int(len(val_ds)) if 'val_ds' in globals() else None,
    'test_windows': int(len(test_ds)) if 'test_ds' in globals() else None,
}

# Extract Domain Info
training_domain = {
    'graph_scope': 'local subgraph',
    'num_nodes_train': int(num_nodes) if 'num_nodes' in globals() else None,
    'num_edges_train': int(edge_index.shape[1]) if 'edge_index' in globals() else None,
    'sensor_nodes_train': int(train_mask.sum().item()) if 'train_mask' in globals() else None,
}

# Consolidate into Master Card
card = {
    'model_name': 'stpignn_local_physics_phase10b_best',
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'metrics': metrics,
    'config': config,
    'data_split': data_split,
    'training_domain': training_domain,
    'artifact_checkpoint': str(ckpt_dir / 'stpignn_local_physics_phase10b_best.pt'),
}

# 1. Save JSON
model_card_json.write_text(json.dumps(card, indent=2), encoding='utf-8')

# 2. Save CSV
flat_row = {**metrics, **config, **data_split, **training_domain}
pd.DataFrame([flat_row]).to_csv(model_card_csv, index=False)

# 3. Save Markdown for Documentation
md_lines = [
    '# STPIGNN Local Physics Model Card (Phase 10B)',
    '',
    '## Metrics',
    f"- best_epoch_by_val: {metrics['best_epoch_by_val']}",
    f"- best_val_scaled_mse: {metrics['best_val_scaled_mse']:.9f}",
    f"- test_scaled_mse: {metrics['test_scaled_mse']:.9f}",
    f"- test_mae_unscaled: {metrics['test_mae_unscaled']:.6f}",
    f"- test_rmse_unscaled: {metrics['test_rmse_unscaled']:.6f}",
    f"- target_scale_q95: {metrics['target_scale_q95']:.6f}",
    '',
    '## Configuration',
    f"- architecture: {config['architecture']}",
    f"- node_in_dim: {config['node_in_dim']}",
    f"- edge_dim: {config['edge_dim']}",
    f"- hidden_dims: spatial={config['spatial_hidden_dim']}, temporal={config['temporal_hidden_dim']}",
    f"- layers: gnn={config['gnn_layers']}, gru={config['gru_layers']}",
    f"- optimizer: {config['optimizer']} (lr={config['learning_rate']}, wd={config['weight_decay']})",
    f"- window: {config['window']}",
    f"- physics_enabled: {config['physics_enabled']}",
    f"- physics_schedule: max_lambda={config['physics_lambda_max']}, warmup_epochs={config['physics_warmup_epochs']}",
    '',
    '## Split',
    f"- split_type: {data_split['split_type']}",
    f"- train_end_t: {data_split['train_end_t']}",
    f"- val_start_t: {data_split['val_start_t']}",
    f"- test_start_t: {data_split['test_start_t']}",
    f"- windows: train={data_split['train_windows']}, val={data_split['val_windows']}, test={data_split['test_windows']}",
    '',
    '## Training Domain',
    f"- graph_scope: {training_domain['graph_scope']}",
    f"- nodes_train: {training_domain['num_nodes_train']}",
    f"- edges_train: {training_domain['num_edges_train']}",
    f"- sensor_nodes_train: {training_domain['sensor_nodes_train']}",
    '',
    '## Artifacts',
    f"- checkpoint: {card['artifact_checkpoint']}",
    f"- json_card: {model_card_json}",
    f"- csv_card: {model_card_csv}",
    '',
    '## Notes',
    '- This model card corresponds to the strict out-of-time Phase 10B result.',
    '- Use this as the baseline reference before Cluster-GCN experiments.'
]

model_card_md.write_text('\n'.join(md_lines), encoding='utf-8')

print('Saved model card JSON:', model_card_json)
print('Saved model card CSV :', model_card_csv)
print('Saved model card MD  :', model_card_md)
print('')
print('Final locked metrics:', metrics)

Saved model card JSON: /home/jupyter-1nt23cb058/Capstone/data/gnn/checkpoints/stpignn_local_physics_phase10b_model_card.json
Saved model card CSV : /home/jupyter-1nt23cb058/Capstone/data/gnn/checkpoints/stpignn_local_physics_phase10b_model_card.csv
Saved model card MD  : /home/jupyter-1nt23cb058/Capstone/data/gnn/checkpoints/stpignn_local_physics_phase10b_model_card.md

Final locked metrics: {'best_epoch_by_val': 15, 'best_val_scaled_mse': 0.0009169053214247383, 'test_scaled_mse': 0.0009470369910732621, 'test_mae_unscaled': 5.080196380615234, 'test_rmse_unscaled': 6.913161277770996, 'target_scale_q95': 224.6432342529297}
